<div align='center'>

# Práctica 7 - Spark

<img src='https://media3.giphy.com/media/v1.Y2lkPTc5MGI3NjExdTZyZDYyeDhwajB3MG9qcmh5bzdlZTJ4dnZmNGh3d2pyMzRtZ3p3dCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/11sN6nMjLT2INO/giphy.gif'>

</div>

### 1)  
Indique cómo queda el **DAG** y qué se ejecuta (y cuántas veces lo hace) en el siguiente script:

```python
rdd1 = sc.parallelize(lista)

for i in range(6):
    rdd2 = rdd1.map(fmap2)
    r1 = rdd2.reduce(fReduce)
    rdd3 = rdd2.union(rdd).map(fmap3) \
                  .reduceByKey(fReduceByKey)

    if (i > 3):
        rdd4 = rdd3.filter(fFilter1).persist()
        print(rdd4.collect())

print(rdd2.collect())
print(rdd3.collect())
```

| Acción           | Cuándo se ejecuta                          | Cuántas veces |
| ---------------- | ------------------------------------------ | ------------- |
| `rdd2.reduce()`  | Dentro del bucle                           | 6 veces       |
| `rdd4.collect()` | Solo cuando `i > 3` → en iteraciones 4 y 5 | 2 veces       |
| `rdd2.collect()` | Fuera del bucle                            | 1 vez         |
| `rdd3.collect()` | Fuera del bucle                            | 1 vez         |

```css

rdd1 → map(fmap2) → reduce(fReduce)
                     ↓
                   union(rdd)
                     ↓
                   map(fmap3)
                     ↓
                reduceByKey(fReduceByKey)
                     ↓
             [si i > 3] filter(fFilter1)
                     ↓
                   collect()
```


### 2)  
Se desea calcular el **promedio de las potencias** de 2 a 5 de los primeros cinco números naturales.

- 1² + 2² + 3² + 4² + 5² = 55 → 55 / 5 = 11  
- 1³ + 2³ + 3³ + 4³ + 5³ = 225 → 225 / 5 = 45  
- 1⁴ + 2⁴ + 3⁴ + 4⁴ + 5⁴ = 979 → 979 / 5 = 195.8  
- 1⁵ + 2⁵ + 3⁵ + 4⁵ + 5⁵ = 4425 → 4425 / 5 = 885  

---

#### ¿Cuál es el error del siguiente script?

```python
rdd = sc.parallelize([1, 2, 3, 4, 5])
for i in range(2, 6):
    acc = sc.broadcast(i)
    rdd = rdd.map(lambda v: v ** acc.value)
    r = rdd.reduce(lambda x, y: x + y)
    r = r / 5
    print(r)
```

❌ Error

El RDD `rdd` se **modifica dentro del bucle**, por lo que en cada iteración se vuelve a elevar al exponente anterior → resultados incorrectos.

✅ Corrección

Mantener el RDD original y aplicar `map` en cada iteración:

```python
rdd = sc.parallelize([1, 2, 3, 4, 5])
for i in range(2, 6):
    acc = sc.broadcast(i)
    r = rdd.map(lambda v: v ** acc.value).reduce(lambda x, y: x + y)
    print(r / 5)
```


---

### 3)  
Plantee un algoritmo que permita aproximarse a la **mediana** de manera **iterativa**, que resulte más eficiente que el método visto en la teoría.

**Idea**

Aproximar la **mediana** de forma **iterativa**, sin ordenar toda la lista.

**Algoritmo propuesto**

1. Calcular promedio inicial.  
2. Contar cuántos valores son menores y mayores.  
3. Ajustar el valor candidato moviéndolo hacia el grupo más grande.  
4. Repetir hasta converger.

```python
rdd = sc.parallelize([5, 2, 9, 4, 7, 6, 3, 8])
med = rdd.mean()   # estimación inicial

for _ in range(10):
    menores = rdd.filter(lambda x: x < med).count()
    mayores = rdd.filter(lambda x: x > med).count()
    if abs(menores - mayores) <= 1:
        break
    med += 0.5 if menores > mayores else -0.5

print("Mediana aproximada:", med)
```

Evita ordenar todo el conjunto → **O(n)** en lugar de **O(n log n)**.

---

### 4)  
Plantee un algoritmo **iterativo** que permita imprimir el **nombre y apellido de los clientes del banco** que tienen un **número primo de cajas de ahorro**.

**Objetivo**

Imprimir el **nombre y apellido** de los clientes que tienen una **cantidad prima de cajas de ahorro**, usando un proceso **iterativo**.

Algoritmo en PySpark
```python
from math import sqrt

def es_primo(n):
    if n < 2: return False
    for i in range(2, int(sqrt(n)) + 1):
        if n % i == 0: return False
    return True

clientes = sc.textFile("Clientes.txt").map(lambda x: x.split("\t"))
# id_cliente, nombre, apellido, dni, fecha, país

cajas = sc.textFile("CajasDeAhorro.txt").map(lambda x: x.split("\t"))
# id_caja, id_cliente, saldo

# Contar cajas por cliente
cant_cajas = cajas.map(lambda x: (x[1], 1)).reduceByKey(lambda a, b: a + b)

# Iterativo: filtrar y mostrar los que tienen número primo de cajas
for i in cant_cajas.collect():
    if es_primo(i[1]):
        nombre = clientes.filter(lambda c: c[0] == i[0]).map(lambda c: (c[1], c[2])).collect()
        print(nombre)
```

El proceso itera sobre los clientes y verifica primalidad sin agrupar todo el dataset en memoria.

---

### 5)  
Plantee un algoritmo **iterativo** que permita resolver el **método de Jacobi**, como el planteado en el **ejercicio 7 de la práctica 2**.

Implementar el **método iterativo de Jacobi** en Spark para resolver un sistema lineal `A·x = b`.

```python
from pyspark import SparkContext

sc = SparkContext("local[*]", "Jacobi")

# Ejemplo de sistema 3x3
A = [[10, -1, 2],
     [-1, 11, -1],
     [2, -1, 10]]
b = [6, 25, -11]
x = [0, 0, 0]  # inicial

rdd = sc.parallelize(list(enumerate(A)))

for _ in range(10):  # número de iteraciones
    x_bc = sc.broadcast(x)
    x_new = rdd.map(lambda row:
        (row[0],
         (b[row[0]] - sum(
            row[1][j] * x_bc.value[j]
            for j in range(len(row[1])) if j != row[0]
         )) / row[1][row[0]])
    ).collect()
    x = [v for _, v in sorted(x_new)]

print("Solución aproximada:", x)
```

* Cada fila de `A` se procesa en paralelo (una por partición).
* Se usa una variable **broadcast** para compartir el vector `x` en cada iteración.
* La condición de parada puede reemplazarse por un umbral de error (`||x_new - x|| < ε`).


---

### 6)  
Dado el dataset **Genealogía**, el cual está formado por:  
`<nombre_individuo, dni_individuo, dni_mamá>`,  
realice distintas funciones que:

---

**a)**  
Dado los DNI de dos individuos, indicar si son **primos**  
(Dos individuos son primos si tienen la misma abuela).


In [1]:
from pyspark.sql import SparkSession
import os, sys

# ===========================
# CONFIGURACIÓN DEL ENTORNO (igual que en EstacionesMeteorologicas)
# ===========================
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# ===========================
# INICIO DE SPARK
# ===========================
spark = SparkSession.builder \
    .appName("GenealogiaPrimos") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

# ===========================
# CARGA DEL DATASET
# ===========================
df = spark.read.csv("../Datasets_spark/Genealogia/Genealogia.txt", sep="\t", header=False)
df = df.toDF("nombre", "dni", "dni_mama")

# 1er join: individuo → mamá
mamas = df.selectExpr("dni as dni_mama", "dni_mama as dni_abuela")

# 2do join: individuo → abuela
df_abuelas = df.join(mamas, on="dni_mama", how="left")

# ===========================
# CONSULTA: ¿tienen la misma abuela?
# ===========================
dni1, dni2 = "4410", "4264"

abuela1 = df_abuelas.filter(df_abuelas["dni"] == dni1).select("dni_abuela").collect()[0][0]
abuela2 = df_abuelas.filter(df_abuelas["dni"] == dni2).select("dni_abuela").collect()[0][0]

print(f"Abuela de {dni1}: {abuela1}")
print(f"Abuela de {dni2}: {abuela2}")

if abuela1 and abuela2 and abuela1 == abuela2:
    print(f"✅ Los individuos {dni1} y {dni2} son primos (abuela común {abuela1})")
else:
    print(f"❌ Los individuos {dni1} y {dni2} NO son primos")

spark.stop()
print("\nEjecución completada ✅")

Abuela de 4410: None
Abuela de 4264: None
✅ Los individuos 4410 y 4264 son primos (abuela común None)

Ejecución completada ✅



---

**b)**  
Dado los DNI de dos individuos `i₁` e `i₂`, indicar si `i₁` es **ancestro** de `i₂`.


In [2]:
from pyspark.sql import SparkSession
import os, sys

# ===========================
# CONFIGURACIÓN DEL ENTORNO
# ===========================
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# ===========================
# INICIO DE SPARK
# ===========================
spark = SparkSession.builder \
    .appName("GenealogiaAncestro") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

# ===========================
# CARGA DEL DATASET
# ===========================
df = spark.read.csv("../Datasets_spark/Genealogia/Genealogia.txt", sep="\t", header=False)
df = df.toDF("nombre", "dni", "dni_mama")

df.show(5, truncate=False)

# ===========================
# FUNCIÓN: verificar si i1 es ancestro de i2
# ===========================
def es_ancestro(df, i1, i2):
    """
    Verifica si i1 es ancestro de i2 en la cadena materna.
    """
    # Iniciamos con la madre directa de i2
    actual = df.filter(df["dni"] == i2).select("dni_mama").collect()
    if not actual or not actual[0][0]:
        return False

    madre_actual = actual[0][0]

    # Recorremos hacia arriba hasta encontrar i1 o quedarnos sin madre
    while madre_actual:
        if madre_actual == i1:
            return True
        fila = df.filter(df["dni"] == madre_actual).select("dni_mama").collect()
        if not fila or not fila[0][0]:
            return False
        madre_actual = fila[0][0]

    return False


# ===========================
# PRUEBA
# ===========================
i1 = "4410"   # posible ancestro
i2 = "4264"   # posible descendiente

resultado = es_ancestro(df, i1, i2)

if resultado:
    print(f"✅ El individuo {i1} ES ancestro de {i2}")
else:
    print(f"❌ El individuo {i1} NO es ancestro de {i2}")

spark.stop()
print("\nEjecución completada ✅")


+--------+----+--------+
|nombre  |dni |dni_mama|
+--------+----+--------+
|Gavgaial|4113|None    |
|Ymxunohg|1536|None    |
|Csmmtmc |1945|None    |
|Lwpodhj |4786|None    |
|Lvnpghoj|979 |None    |
+--------+----+--------+
only showing top 5 rows

❌ El individuo 4410 NO es ancestro de 4264

Ejecución completada ✅


In [ ]:
---

**c)**  
Obtener el **nombre de la “abuela”** que tiene **más descendientes**.

In [5]:
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("GenealogiaAbuela") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

# ===========================
# CARGA DEL DATASET
# ===========================
df = spark.read.csv("../Datasets_spark/Genealogia/Genealogia.txt", sep="\t", header=False)
df = df.toDF("nombre", "dni", "dni_mama")

# ===========================
# PRIMER JOIN: individuo → mamá
# ===========================
mamas = df.select(
    F.col("dni").alias("dni_mama"),
    F.col("dni_mama").alias("dni_abuela")
)

# join para obtener la abuela de cada individuo
df_abuelas = df.join(
    df.select(F.col("dni").alias("dni_mama"), F.col("dni_mama").alias("dni_abuela")),
    on="dni_mama",
    how="left"
)

# ===========================
# OBTENER NOMBRE DE LA ABUELA CON MÁS DESCENDIENTES
# ===========================
# join con el nombre de la abuela
df_abuelas = df_abuelas.join(
    df.select(F.col("dni").alias("dni_abuela"), F.col("nombre").alias("nombre_abuela")),
    on="dni_abuela",
    how="left"
)

# contar descendientes por abuela
abuela_counts = (
    df_abuelas
    .filter(F.col("dni_abuela").isNotNull())
    .groupBy("dni_abuela", "nombre_abuela")
    .count()
    .orderBy(F.desc("count"))
)

abuela_top = abuela_counts.limit(1).collect()[0]
print(f"👵 La abuela con más descendientes es {abuela_top['nombre_abuela']} (DNI {abuela_top['dni_abuela']}) con {abuela_top['count']} descendientes.")

spark.stop()
print("\nEjecución completada ✅")

👵 La abuela con más descendientes es None (DNI None) con 10 descendientes.

Ejecución completada ✅


---

**d)**  
Listar los **nombres de los hermanos** de la **familia más numerosa**  
(la cantidad de integrantes de una familia se calcula con la cantidad de hermanos más la mamá).  
Podría existir más de una familia más numerosa; en ese caso, imprimir los nombres de todos los hermanos integrantes de cada familia.


In [7]:
spark = SparkSession.builder \
    .appName("GenealogiaFamiliaNumerosa") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

# ===========================
# CARGA DEL DATASET
# ===========================
df = spark.read.csv("../Datasets_spark/Genealogia/Genealogia.txt", sep="\t", header=False)
df = df.toDF("nombre", "dni", "dni_mama")

# ===========================
# AGRUPAR POR MAMÁ PARA OBTENER CANTIDAD DE HIJOS
# ===========================
familias = (
    df.groupBy("dni_mama")
      .agg(
          F.collect_list("nombre").alias("hijos"),
          F.count("nombre").alias("cant_hijos")
      )
      .filter(F.col("dni_mama").isNotNull())
)

# Agregar +1 por la mamá → total integrantes
familias = familias.withColumn("total_integrantes", F.col("cant_hijos") + 1)

# ===========================
# OBTENER FAMILIA(S) MÁS NUMEROSA(S)
# ===========================
max_integrantes = familias.agg(F.max("total_integrantes")).collect()[0][0]
familias_max = familias.filter(F.col("total_integrantes") == max_integrantes)

# ===========================
# UNIR CON df PARA OBTENER NOMBRE DE LA MAMÁ
# ===========================
familias_max = familias_max.join(
    df.select(F.col("dni").alias("dni_mama"), F.col("nombre").alias("nombre_mama")),
    on="dni_mama",
    how="left"
)

# ===========================
# MOSTRAR RESULTADOS
# ===========================
familias_max.select("nombre_mama", "hijos", "total_integrantes").show(truncate=False)

# Imprimir de forma más legible
rows = familias_max.collect()
for row in rows:
    print(f"👩 Mamá: {row['nombre_mama']} (Integrantes: {row['total_integrantes']})")
    print("👧 Hijos:", ", ".join(row['hijos']))
    print("—" * 50)

spark.stop()
print("\nEjecución completada ✅")

+-----------+-------------------------------------------------------------------------------------------+-----------------+
|nombre_mama|hijos                                                                                      |total_integrantes|
+-----------+-------------------------------------------------------------------------------------------+-----------------+
|NULL       |[Gavgaial, Ymxunohg, Csmmtmc, Lwpodhj, Lvnpghoj, Dwnuphn, Nmnfqa, Cyumep, Lpclex, Huoaczgp]|11               |
+-----------+-------------------------------------------------------------------------------------------+-----------------+

👩 Mamá: None (Integrantes: 11)
👧 Hijos: Gavgaial, Ymxunohg, Csmmtmc, Lwpodhj, Lvnpghoj, Dwnuphn, Nmnfqa, Cyumep, Lpclex, Huoaczgp
——————————————————————————————————————————————————

Ejecución completada ✅
